# Purpose: Creating coordinate files for promoters and genes
Taking RefSeq coordinates for genes and parsing out: 
1. Promoter coordinates 
2. Gene coordinates for
each unique TSS and TES (alternative splicing isoforms are ignored) 
3. One set of coordinates for each gene body, using the most internal TSS and TES, and removing the first and last 1 kb so as to remove sites of
polymerase pausing

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns


#amount upstream and downstream of annotated TSS to consider a promoter
lowerBound=1000
upperBound=1000

chromList = ['chrX']
chromList = chromList + [f'chr{i}' for i in range(1,23)]


In [2]:
# input
refseqFile = '../../Manuscript_data/hg38_refseq.txt'
outputs = '../../figure_outputs/'

# outputs

#BED file that has one of coordinate set per gene, 
#using most internal TSS and TES, removing 1 kb from 5' and 3' ends
gbFile=outputs+'hg38_refseq_NR_gene_bodies.bed' 

promoterFile=outputs+'hg38_refseq_promoters.bed'

In [3]:
rs_df = pd.read_csv(refseqFile, sep='\t')
rs_df

,#bin,name,chrom,strand,txStart,txEnd,cdsStart,cdsEnd,exonCount,exonStarts,exonEnds,score,name2,cdsStartStat,cdsEndStat,exonFrames
0,0,XM_011541469.1,chr1,-,67092175,67109072,67093004,67103382,5,"67092175,67095234,67096251,67103237,67109028,","67093604,67095421,67096321,67103382,67109072,",0,C1orf141,cmpl,cmpl,"0,2,1,0,-1,"
1,0,XM_011541467.1,chr1,-,67092175,67131183,67093004,67127240,9,"67092175,67095234,67096251,67103237,67111576,6...","67093604,67095421,67096321,67103343,67111644,6...",0,C1orf141,cmpl,cmpl,"0,2,1,0,1,2,0,0,-1,"
2,0,XM_017001276.1,chr1,-,67092175,67131227,67093004,67127240,9,"67092175,67095234,67096251,67103237,67111576,6...","67093604,67095421,67096321,67103382,67111644,6...",0,C1orf141,cmpl,cmpl,"0,2,1,0,1,2,0,0,-1,"
3,0,XM_011541465.2,chr1,-,67092175,67134962,67093004,67127240,9,"67092175,67095234,67096251,67103237,67111576,6...","67093604,67095421,67096321,67103382,67111644,6...",0,C1orf141,cmpl,cmpl,"0,2,1,0,1,2,0,0,-1,"
4,0,NR_075077.1,chr1,-,67092175,67134971,67134971,67134971,10,"67092175,67096251,67103237,67111576,67113613,6...","67093604,67096321,67103382,67111644,67113756,6...",0,C1orf141,none,none,"-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
167464,586,XM_017030167.1,chr22_KI270734v1_random,-,138085,153840,138479,152899,13,"138085,138742,142193,143613,144748,145003,1466...","138667,138831,142292,143789,144895,145096,1467...",0,LOC102724788,cmpl,cmpl,"1,2,2,0,0,0,2,0,0,0,1,1,0,"
167465,586,XR_951398.2,chr22_KI270734v1_random,-,138085,161588,161588,161588,14,"138085,138742,142193,143613,144748,145003,1466...","138667,138831,142292,143789,144929,145096,1467...",0,LOC102724788,none,none,"-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,"
167466,586,XM_017030168.1,chr22_KI270734v1_random,-,138085,161592,138479,156446,14,"138085,138742,142193,143613,144748,145003,1466...","138667,138831,142292,143789,144895,145096,1467...",0,LOC102724788,cmpl,cmpl,"1,2,2,0,0,0,2,0,0,1,1,2,0,-1,"
167467,586,XM_006724936.3,chr22_KI270734v1_random,-,138085,161594,138479,150995,13,"138085,138742,142193,143613,144748,145003,1466...","138667,138831,142292,143789,144895,145096,1467...",0,LOC102724788,cmpl,cmpl,"1,2,2,0,0,0,2,0,0,0,0,-1,-1,"


In [4]:
# keep NM transcripts in typical chromosomes
nm_mask = rs_df['name'].str.split('_', expand=True)[0] == 'NM'
rs_df = rs_df.loc[nm_mask]
rs_df = rs_df.loc[rs_df['chrom'].isin(chromList)].reset_index(drop=True)
rs_df

,#bin,name,chrom,strand,txStart,txEnd,cdsStart,cdsEnd,exonCount,exonStarts,exonEnds,score,name2,cdsStartStat,cdsEndStat,exonFrames
0,0,NM_001276352.1,chr1,-,67092175,67134971,67093579,67127240,9,"67092175,67096251,67103237,67111576,67115351,6...","67093604,67096321,67103382,67111644,67115464,6...",0,C1orf141,cmpl,cmpl,"2,1,0,1,2,0,0,-1,-1,"
1,0,NM_001276351.1,chr1,-,67092175,67134971,67093004,67127240,8,"67092175,67095234,67096251,67115351,67125751,6...","67093604,67095421,67096321,67115464,67125909,6...",0,C1orf141,cmpl,cmpl,"0,2,1,2,0,0,-1,-1,"
2,0,NM_001005337.2,chr1,+,201283451,201332993,201283702,201328836,14,"201283451,201293941,201313165,201316552,201317...","201283904,201294045,201313560,201316697,201317...",0,PKP1,cmpl,cmpl,"0,1,0,2,0,1,2,0,0,0,1,2,0,-1,"
3,0,NM_000299.3,chr1,+,201283451,201332993,201283702,201328836,15,"201283451,201293941,201313165,201316552,201317...","201283904,201294045,201313560,201316697,201317...",0,PKP1,cmpl,cmpl,"0,1,0,2,0,1,2,2,0,0,0,1,2,0,-1,"
4,1,NM_001042682.1,chr1,-,8352403,8423687,8355086,8364133,13,"8352403,8355418,8356099,8358195,8359763,836011...","8355120,8355599,8356246,8358916,8359986,836149...",0,RERE,cmpl,cmpl,"2,1,1,0,2,0,0,0,0,-1,-1,-1,-1,"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49860,972,NM_001130922.2,chr22,-,50767491,50783682,50768775,50782294,9,"50767491,50769040,50769454,50769904,50775771,5...","50768874,50769124,50769549,50770016,50775851,5...",0,RABL2B,cmpl,cmpl,"0,0,1,0,1,2,2,0,-1,"
49861,972,NM_001350010.1,chr22,-,50767491,50783682,50768775,50782294,9,"50767491,50769040,50769424,50769904,50775771,5...","50768874,50769124,50769552,50770016,50775851,5...",0,RABL2B,cmpl,cmpl,"0,0,1,0,1,2,2,0,-1,"
49862,972,NM_001130919.2,chr22,-,50767491,50783682,50768775,50782294,9,"50767491,50769040,50769454,50769904,50775771,5...","50768874,50769124,50769552,50770016,50775851,5...",0,RABL2B,cmpl,cmpl,"0,0,1,0,1,2,2,0,-1,"
49863,972,NM_001350015.1,chr22,-,50767491,50783682,50768775,50782294,9,"50767491,50769040,50769424,50769904,50775771,5...","50768874,50769124,50769549,50770016,50775851,5...",0,RABL2B,cmpl,cmpl,"0,0,1,0,1,2,2,0,-1,"


In [5]:
rs_bed = rs_df[['chrom','txStart','txEnd','name2']]
rs_bed['score'] = '.'
rs_bed['strand'] = rs_df['strand']
rs_bed

/var/folders/x2/34lg9m394nj1zx1ph2bkj0tw0000gn/T/ipykernel_37069/546778032.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rs_bed['score'] = '.'
/var/folders/x2/34lg9m394nj1zx1ph2bkj0tw0000gn/T/ipykernel_37069/546778032.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rs_bed['strand'] = rs_df['strand']


,chrom,txStart,txEnd,name2,score,strand
0,chr1,67092175,67134971,C1orf141,.,-
1,chr1,67092175,67134971,C1orf141,.,-
2,chr1,201283451,201332993,PKP1,.,+
3,chr1,201283451,201332993,PKP1,.,+
4,chr1,8352403,8423687,RERE,.,-
...,...,...,...,...,...,...
49860,chr22,50767491,50783682,RABL2B,.,-
49861,chr22,50767491,50783682,RABL2B,.,-
49862,chr22,50767491,50783682,RABL2B,.,-
49863,chr22,50767491,50783682,RABL2B,.,-


In [60]:
genes = list(rs_bed['name2'].unique())
rs_bed_gb = pd.DataFrame(np.zeros((len(genes),6)))
promoters = []


In [61]:
def gb_coords(df, lowerBound, upperBound, gene):
    chr = df['chrom'].unique()
    start = df['txStart'].max() + lowerBound
    end = df['txEnd'].min() - upperBound
    strand = df['strand'].unique()
    
    return pd.Series((chr, start, end,gene, '.', strand))

In [62]:
def prom_coords(df, lowerBound, upperBound, gene):
    chr = df['chrom'].unique()[0]
    strand = df['strand'].unique()[0]
    if strand == '+':
        starts = df['txStart'].unique()
    else:
        starts = df['txEnd'].unique()
    toReturn = []
    for start in starts:
        toReturn.append([chr, start - lowerBound, start + upperBound, gene, '.', strand])
    return toReturn

In [63]:
for i, gene in enumerate(genes):
    gene_mask = rs_bed['name2'] == gene

    rs_bed_gb.iloc[i,:] = gb_coords(
        rs_bed.loc[gene_mask],
        lowerBound,
        upperBound,
        gene
    )
    
    promoters = promoters + prom_coords(
        rs_bed.loc[gene_mask],
        lowerBound,
        upperBound,
        gene
    )



    

/var/folders/x2/34lg9m394nj1zx1ph2bkj0tw0000gn/T/ipykernel_37069/2712326033.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['chr1']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  rs_bed_gb.iloc[i,:] = gb_coords(
/var/folders/x2/34lg9m394nj1zx1ph2bkj0tw0000gn/T/ipykernel_37069/2712326033.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'C1orf141' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  rs_bed_gb.iloc[i,:] = gb_coords(
/var/folders/x2/34lg9m394nj1zx1ph2bkj0tw0000gn/T/ipykernel_37069/2712326033.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '.' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  rs_be

In [64]:
pd.DataFrame(promoters)

,0,1,2,3,4,5
0,chr1,67133971,67135971,C1orf141,.,-
1,chr1,201282451,201284451,PKP1,.,+
2,chr1,8422687,8424687,RERE,.,-
3,chr1,8816640,8818640,RERE,.,-
4,chr1,34164274,34166274,CSMD2,.,-
...,...,...,...,...,...,...
26445,chr22,50581999,50583999,CHKB,.,-
26446,chr22,50599684,50601684,MAPK8IP2,.,+
26447,chr22,50627173,50629173,ARSA,.,-
26448,chr22,50737223,50739223,ACR,.,+


In [65]:
rs_bed_gb

,0,1,2,3,4,5
0,chr1,67093175.0,67133971.0,C1orf141,.,-
1,chr1,201284451.0,201331993.0,PKP1,.,+
2,chr1,8353403.0,8422687.0,RERE,.,-
3,chr1,33514998.0,34164274.0,CSMD2,.,-
4,chr1,75207389.0,75610114.0,SLC44A5,.,-
...,...,...,...,...,...,...
19375,chr22,50579957.0,50581999.0,CHKB,.,-
19376,chr22,50601684.0,50612981.0,MAPK8IP2,.,+
19377,chr22,50623753.0,50627173.0,ARSA,.,-
19378,chr22,50739223.0,50744299.0,ACR,.,+
